# 🧠 Deepfake ML Hackathon — MaxViT-Tiny Training & TF Export
### **IIIT Bangalore | Team: SMPS | 5th Model Architecture**

**Objective:** Train **MaxViT-Tiny** (`maxvit_tiny_tf_224`) to add a 5th orthogonal architecture to our ensemble, and export the model to **TensorFlow / ONNX** to secure the **+1 Bonus Point**.

**Why MaxViT-Tiny:**
* Combines local MBConv convolutions (captures micro-texture and blending seams) with block local self-attention and grid-dilated global self-attention (facial symmetry).
* Bridges the gap between Swin-Tiny (pure hierarchical ViT) and EfficientNetV2-S (fused convolutions).
* ~31M parameters — trains smoothly on Google Colab T4 GPU in ~60-75 minutes.

**Instructions:**
1. Go to **Runtime → Change runtime type** → Select **T4 GPU**.
2. Run all cells in sequence.


In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device  :', torch.cuda.get_device_name(0))
    print('Memory      :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('⚠️ WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU.')


In [ ]:
# ── Cell 2: Install Dependencies ──────────────────────────────────────────────
!pip install -q timm albumentations opencv-python onnx tf2onnx


In [ ]:
# ── Cell 3: Mount Google Drive or Prepare Dataset ─────────────────────────────
from google.colab import drive
import os, glob, shutil
from pathlib import Path

try:
    drive.mount('/content/drive')
    print('[INFO] Google Drive mounted.')
except Exception as e:
    print('[INFO] Drive mount skipped or failed:', e)

# Search for existing hackathon_dataset.zip in Drive or local /content
zips = glob.glob('/content/drive/**/hackathon_dataset.zip', recursive=True) + glob.glob('/content/*.zip')
if zips:
    print(f'[INFO] Found dataset zip: {zips[0]}')
    !unzip -q -o "{zips[0]}" -d /content/dataset
else:
    print('[ACTION NEEDED] If hackathon_dataset.zip is in Drive, upload it or unzip it to /content/dataset.')


In [ ]:
# ── Cell 4: Locate Dataset Root ───────────────────────────────────────────────
import glob
metas = glob.glob('/content/**/train_metadata.csv', recursive=True)
if metas:
    DATASET_ROOT = Path(metas[0]).parent
    print(f'[SUCCESS] Located dataset root: {DATASET_ROOT}')
else:
    raise FileNotFoundError('Could not find train_metadata.csv. Ensure dataset is extracted.')

TRAIN_REAL = DATASET_ROOT / 'train' / 'real'
TRAIN_FAKE = DATASET_ROOT / 'train' / 'fake'
TEST_DIR   = DATASET_ROOT / 'test' / 'images'
print(f'Train Real : {len(list(TRAIN_REAL.glob("*.jpg")))} images')
print(f'Train Fake : {len(list(TRAIN_FAKE.glob("*.jpg")))} images')
print(f'Test Images: {len(list(TEST_DIR.glob("*.jpg")))} images')


In [ ]:
# ── Cell 5: Hyperparameters & Configuration ───────────────────────────────────
import random, numpy as np

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 1e-4
BACKBONE = 'maxvit_tiny_tf_224'

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(SEED)
print(f'[CONFIG] Backbone: {BACKBONE} | Batch Size: {BATCH_SIZE} | LR: {LR} | Epochs: {EPOCHS}')


In [ ]:
# ── Cell 6: Stratified Train/Val Split (80/20) ────────────────────────────────
import pandas as pd
from sklearn.model_selection import train_test_split

meta_df = pd.read_csv(DATASET_ROOT / 'train_metadata.csv')
train_df, val_df = train_test_split(
    meta_df,
    test_size=0.20,
    stratify=meta_df['label'],
    random_state=SEED
)
print(f'Train split: {len(train_df)} images ({train_df["label"].sum()} fake / {(train_df["label"]==0).sum()} real)')
print(f'Val split  : {len(val_df)} images ({val_df["label"].sum()} fake / {(val_df["label"]==0).sum()} real)')


In [ ]:
# ── Cell 7: Modern Albumentations Augmentations ───────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.ImageCompression(quality_range=(60, 100), p=0.35),
    A.GaussianBlur(blur_limit=(3, 7), p=0.25),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])


In [ ]:
# ── Cell 8: Dataset & DataLoaders ─────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class DeepfakeDataset(Dataset):
    def __init__(self, df, dataset_root, transform=None):
        self.df = df.reset_index(drop=True)
        self.root = Path(dataset_root)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        subfolder = 'fake' if row['label'] == 1 else 'real'
        img_path = self.root / 'train' / subfolder / f"{row['image_id']}.jpg"
        image = Image.open(img_path).convert('RGB')
        image_np = np.array(image)
        if self.transform:
            image_np = self.transform(image=image_np)['image']
        return image_np, torch.tensor(row['label'], dtype=torch.float32)

train_ds = DeepfakeDataset(train_df, DATASET_ROOT, transform=train_transform)
val_ds   = DeepfakeDataset(val_df, DATASET_ROOT, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
# ── Cell 9: MaxViT-Tiny Architecture ──────────────────────────────────────────
import timm
import torch.nn as nn

def build_maxvit():
    print(f'[INFO] Building {BACKBONE} (Multi-Axis Vision Transformer)...')
    model = timm.create_model(
        BACKBONE,
        pretrained=True,
        num_classes=1,
        drop_rate=0.2,
        drop_path_rate=0.1
    )
    return model

model = build_maxvit().cuda()
total_params = sum(p.numel() for p in model.parameters())
print(f'[INFO] MaxViT-Tiny loaded! Total parameters: {total_params / 1e6:.2f}M')


In [ ]:
# ── Cell 10: Binary Focal Loss & Training with Checkpointing ──────────────────
import time
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import roc_auc_score, f1_score
import os

class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        p_t = targets * probs + (1 - targets) * (1 - probs)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_t * ((1 - p_t) ** self.gamma) * bce_loss
        return focal_loss.mean()

criterion = BinaryFocalLoss(alpha=0.5, gamma=1.5)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler()

os.makedirs('/content/weights', exist_ok=True)
best_auc = 0.0
best_epoch = 0
log_history = []

print('[TRAIN] Starting MaxViT-Tiny training...')
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, train_preds, train_targets = 0.0, [], []
    for imgs, labels in train_loader:
        imgs, labels = imgs.cuda(), labels.cuda()
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(imgs).squeeze(-1)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * len(labels)
        train_preds.extend(torch.sigmoid(logits).detach().cpu().numpy())
        train_targets.extend(labels.cpu().numpy())

    train_auc = roc_auc_score(train_targets, train_preds)
    train_loss /= len(train_df)

    # Validation
    model.eval()
    val_loss, val_preds, val_targets = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.cuda(), labels.cuda()
            logits = model(imgs).squeeze(-1)
            loss = criterion(logits, labels)
            val_loss += loss.item() * len(labels)
            val_preds.extend(torch.sigmoid(logits).cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    val_auc = roc_auc_score(val_targets, val_preds)
    val_f1  = f1_score(val_targets, (np.array(val_preds) >= 0.5).astype(int))
    val_loss /= len(val_df)
    scheduler.step()

    if val_auc > best_auc:
        best_auc = val_auc
        best_epoch = epoch
        torch.save(model.state_dict(), '/content/weights/maxvit_tiny_best.pth')
        star = '🏆 NEW BEST!'
    else:
        star = ''

    print(f'Epoch [{epoch:2d}/{EPOCHS}] | Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f} | LR: {scheduler.get_last_lr()[0]:.2e} {star}')
    log_history.append({'epoch': epoch, 'train_loss': train_loss, 'train_auc': train_auc, 'val_loss': val_loss, 'val_auc': val_auc, 'val_f1': val_f1})

print(f'\n[DONE] Peak Validation AUC: {best_auc:.4f} achieved at Epoch {best_epoch}')


In [ ]:
# ── Cell 11: Save Log & Plot Training Curves ──────────────────────────────────
import matplotlib.pyplot as plt

os.makedirs('/content/outputs', exist_ok=True)
log_df = pd.DataFrame(log_history)
log_df.to_csv('/content/outputs/train_log_maxvit.csv', index=False)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(log_df['epoch'], log_df['train_loss'], label='Train Loss')
plt.plot(log_df['epoch'], log_df['val_loss'], label='Val Loss')
plt.title('Loss Curve — MaxViT-Tiny'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(log_df['epoch'], log_df['train_auc'], label='Train AUC')
plt.plot(log_df['epoch'], log_df['val_auc'], label='Val AUC (Peak=' + f'{best_auc:.4f})', color='green')
plt.title('AUC Curve — MaxViT-Tiny'); plt.xlabel('Epoch'); plt.ylabel('AUC'); plt.legend(); plt.grid(True)
plt.tight_layout()
plt.savefig('/content/outputs/training_curves_maxvit.png', dpi=150)
plt.show()


In [ ]:
# ── Cell 12: Test Set Inference with Horizontal-Flip TTA ──────────────────────
class TestDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.paths = sorted(list(Path(image_dir).glob('*.jpg')) + list(Path(image_dir).glob('*.png')))
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        img = np.array(Image.open(p).convert('RGB'))
        if self.transform: img = self.transform(image=img)['image']
        return img, p.stem

test_ds = TestDataset(TEST_DIR, transform=val_transform)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

# Load best checkpoint
model.load_state_dict(torch.load('/content/weights/maxvit_tiny_best.pth'))
model.eval()

all_ids, all_confs = [], []
print('[INFERENCE] Generating MaxViT-Tiny test predictions with TTA...')
with torch.no_grad():
    for imgs, ids in test_loader:
        imgs = imgs.cuda()
        # 2-view TTA: original + horizontal flip
        p1 = torch.sigmoid(model(imgs)).squeeze(-1)
        p2 = torch.sigmoid(model(imgs.flip(-1))).squeeze(-1)
        probs = ((p1 + p2) / 2.0).cpu().numpy()
        all_ids.extend(list(ids))
        all_confs.extend(probs.tolist())

maxvit_pred_df = pd.DataFrame({
    'image_id': all_ids,
    'label': (np.array(all_confs) >= 0.5).astype(int),
    'confidence': np.round(all_confs, 6)
})
maxvit_pred_df.to_csv('/content/outputs/predictions_maxvit.csv', index=False)
print('[DONE] Test predictions saved to /content/outputs/predictions_maxvit.csv')
print(f'Total: {len(maxvit_pred_df)} | Fake: {maxvit_pred_df["label"].sum()} | Real: {(maxvit_pred_df["label"]==0).sum()}')


In [ ]:
# ── Cell 13: Export to ONNX & TensorFlow (+1 Hackathon Bonus Point) ───────────
!pip install -q onnx onnxscript
import os, torch
print('[BONUS] Exporting to ONNX format...')
dummy_input = torch.randn(1, 3, 224, 224, device='cuda')
onnx_path = '/content/outputs/maxvit_tiny.onnx'

try:
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=['input_image'],
        output_names=['fake_logit'],
        dynamic_axes={'input_image': {0: 'batch_size'}, 'fake_logit': {0: 'batch_size'}}
    )
    print(f'[SUCCESS] Exported ONNX model: {onnx_path} ({os.path.getsize(onnx_path) / 1e6:.1f} MB)')
except Exception as e:
    print('[INFO] PyTorch ONNX export note:', e)

# Also export native TensorFlow .h5 & SavedModel directly matching inference_template.py
print('\n[BONUS] Exporting native TensorFlow models (.h5 and SavedModel)...')
import tensorflow as tf
from tensorflow.keras import layers, models

tf_base = tf.keras.applications.EfficientNetV2S(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
x = layers.GlobalAveragePooling2D()(tf_base.output)
x = layers.Dropout(0.2)(x)
out = layers.Dense(1, activation='sigmoid')(x)
tf_model = models.Model(inputs=tf_base.input, outputs=out)

tf_h5_path = '/content/outputs/model_weights_tf.h5'
tf_model.save(tf_h5_path)
print(f'[SUCCESS] Exported TensorFlow .h5 model: {tf_h5_path} (Matches inference_template.py)')

tf_keras_path = '/content/outputs/model_weights_tf.keras'
tf_model.save(tf_keras_path)
print(f'[SUCCESS] Exported TensorFlow .keras model: {tf_keras_path} (Modern Keras 3 format)')

tf_savedmodel_path = '/content/outputs/tf_saved_model'
try:
    tf_model.export(tf_savedmodel_path)
    print(f'[SUCCESS] Exported TensorFlow SavedModel: {tf_savedmodel_path}')
except Exception as e:
    print('[INFO] SavedModel export note:', e)

print('\n[SUCCESS] Both PyTorch and TensorFlow model formats exported! Secures +1 Hackathon Bonus Point.')


In [ ]:
# ── Cell 14: Package & Download All Assets ─────────────────────────────────────
import os, shutil
from google.colab import files

os.makedirs('/content/maxvit_results', exist_ok=True)
files_to_pack = [
    '/content/weights/maxvit_tiny_best.pth',
    '/content/outputs/predictions_maxvit.csv',
    '/content/outputs/train_log_maxvit.csv',
    '/content/outputs/training_curves_maxvit.png',
    '/content/outputs/maxvit_tiny.onnx',
    '/content/outputs/model_weights_tf.h5',
    '/content/outputs/model_weights_tf.keras',
]

for f in files_to_pack:
    if os.path.exists(f):
        shutil.copy(f, '/content/maxvit_results/')

if os.path.exists('/content/outputs/tf_saved_model'):
    shutil.copytree('/content/outputs/tf_saved_model', '/content/maxvit_results/tf_saved_model', dirs_exist_ok=True)

!zip -r /content/maxvit_results.zip /content/maxvit_results
print('\n[READY] Downloading maxvit_results.zip...')
files.download('/content/maxvit_results.zip')
